In [1]:
# NLP303 Assessment 3
#
# Detecting AI-generated Text
# Using Transformer-Based Classification
#
#
# Jonathan Lim - A00142089
#
# Thomas Galindo Salazar - A00129258
#
# Tibor Titusz Tarcsai - A00121308
#
#
#
# The purpose of this implementation is to demonstrate the building of a working prototype for a Classification task based on the Assessment 2 proposal.
#
#
#***********************
# HOW TO RUN THE CODE:
#***********************
#
# 1. - The notebook can run either in Jupyter Notebook or Google Colab
#    - If using Google Colab, a Google Drive account is required for data access and storage.
#      Link for running on Colab:
#
# 2. All helper functions are imported and loaded from the src folder
#

# Environment Setup and Dependencies

In [1]:
# Clones Project from Github
!git clone -b colab https://github.com/Titusz87/roberta-ai-text-detector.git

# Sets working directory for Colab
%cd /content/roberta-ai-text-detector/notebook/

Cloning into 'roberta-ai-text-detector'...
remote: Enumerating objects: 177, done.
remote: Counting objects: 100% (177/177), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 177 (delta 88), reused 127 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (177/177), 94.31 KiB | 3.77 MiB/s, done.
Resolving deltas: 100% (88/88), done.
/content/roberta-ai-text-detector/notebook


In [2]:
!pip install -r ../requirements.txt
print("\n\n### Dependencies installed successfully ###")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 117.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 122.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 127.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: gdown
    Found existing installation: gdown 5.2.2
    Uninstalling gdown-5.2.2:
      Successfully uninstalled gdown-5.2.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Un

# Section 1 - Load Dataset

In [3]:
# Initialises the downloader class
import sys
sys.path.append("../")

from src.utils.downloader import Downloader

downloader = Downloader()

# Downloads raw datasets from google drive
print("### Downloading Dataset from Google Drive... ###")
downloader.start_downloading_dataset()
print(f"\n\nDatasets are downloaded and and located at the 'data/' folder.")

### Downloading Dataset from Google Drive... ###


Downloading...
From: https://drive.google.com/uc?id=1v8ZKV3p6KLDMsOscVLj1Z5zgNYGJfaQp
To: /content/roberta-ai-text-detector/data/humanised_v2_first_2400.csv
100%|██████████| 7.48M/7.48M [00:00<00:00, 93.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1U9Lhpo2qet7dPAuswxHFGAJEggc26bR0
To: /content/roberta-ai-text-detector/data/ai_polished_v2_first_2400.csv
100%|██████████| 7.36M/7.36M [00:00<00:00, 40.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=182-e58HGw67tacTudS7wacm6DZZCbMEZ
To: /content/roberta-ai-text-detector/data/pure_ai_v2_first_2400.csv
100%|██████████| 4.85M/4.85M [00:00<00:00, 55.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=15BDFQcaylNmK6Uy-BaKKe__jXQ6MgdmK
To: /content/roberta-ai-text-detector/data/pure_human_v2_first_2400.csv
100%|██████████| 3.96M/3.96M [00:00<00:00, 83.3MB/s]



Datasets are downloaded and and located at the 'data/' folder.


In [10]:
# Builds the dataset
from src.datasetbuilder import DatasetBuilder

builder = DatasetBuilder()

raw_dataset = builder.build_dataset(
    builder.raw_dataset_paths[0]["pure_human"],     # Label 0
    builder.raw_dataset_paths[1]["pure_ai"],        # Label 1
    builder.raw_dataset_paths[2]["ai_polished"],    # Label 2
    builder.raw_dataset_paths[3]["humanised"],      # Label 3
)

print(len(raw_dataset))

9600


# Section 2 - Text Preprocessing

In [11]:
# Drops rows where 'text' is missing
raw_dataset = raw_dataset.dropna(subset=["text"]).reset_index(drop=True)

print(len(raw_dataset))

9600


In [12]:
# Initialises the TextPreprocessor class for text cleaning
from src.utils.preprocessing import TextPreprocessor

text_preprocessor = TextPreprocessor()

raw_dataset["cleaned_text"] = raw_dataset["text"].apply(text_preprocessor.clean_text)



print(raw_dataset[["cleaned_text", "text"]].iloc[7204])

cleaned_text    The introduction of two new scheduling primiti...
text            \nThe introduction of two new scheduling primi...
Name: 7204, dtype: object


# Section 3 - Data Split

In [13]:
# Shuffles the dataset and stores features and labels
dataset_shuffled = raw_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

features = dataset_shuffled['cleaned_text']
labels = dataset_shuffled['label']


# Splits the data into an initial 80-20 split
from sklearn.model_selection import train_test_split

x_train_raw, x_temp_raw, y_train, y_temp = train_test_split(
    features,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

# Split the remaining data into validation and test (10-10)
x_val_raw, x_test_raw, y_val, y_test = train_test_split(
    x_temp_raw,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# Prints class distributions

print("\n * Training Labels (y_train):")

print(y_train.value_counts().sort_index())

print("\n * Validation Labels (y_val):")

print(y_val.value_counts().sort_index())

print("\n * Testing Labels (y_test):")

print(y_test.value_counts().sort_index())


 * Training Labels (y_train):
label
0    1920
1    1920
2    1920
3    1920
Name: count, dtype: int64

 * Validation Labels (y_val):
label
0    240
1    240
2    240
3    240
Name: count, dtype: int64

 * Testing Labels (y_test):
label
0    240
1    240
2    240
3    240
Name: count, dtype: int64


# Section 3 - Tokenisation (BPE)

In [14]:

"""""
In this section during tokenisation the encodings are created to suit the basemodel with 512 token limit.
"""""

# Imports tokeniser
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("fakespot-ai/roberta-base-ai-text-detection-v1")

# Creates input encodings
train_encodings = tokenizer(x_train_raw.to_list(), truncation=True, padding="max_length", max_length=512)
val_encodings = tokenizer(x_val_raw.to_list(), truncation=True, padding="max_length", max_length=512)
test_encodings = tokenizer(x_test_raw.to_list(), truncation=True, padding="max_length", max_length=512)

# Transforms encodings into Pytorch tensors
from src.dataset_wrapper import DatasetWrapper

X_train = DatasetWrapper(train_encodings, y_train)
X_val = DatasetWrapper(val_encodings, y_val)
X_test = DatasetWrapper(test_encodings, y_test)


# Prints the first sample's token IDs, mask, and label
X_test[0]




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

{'input_ids': tensor([    0,  2522,  1548, 15678,  9037,  4484,    19,  1501, 44871,  6448,
             7,  6292,   258,  2078,     8,  8611,     6,   150,    67,  6477,
         43797,  4983,     4,   166,  1400,   375,   775,    25, 41834, 26070,
           634, 10160, 42472,   256, 46368, 32146,    36, 16972, 13123,   322,
            83,    92,  6779,  5448,  2386,   201,     7,  2935, 12535,   148,
          7356,     6,  1195,    87, 13304,    15,  2052, 10576,    59,  4298,
         17294,    50,  2065, 25212,  7373,    14,  6876,  5891, 27229,   227,
         16420,     6, 15515,    86,   335,     4,   152,  3315,     7,  5849,
         20477,    81,   349, 27304,     6,    61,    64, 26679, 33345,  3867,
            52,  6581,   943, 25083,   396,  3625,  2284, 38163, 13879,     4,
           635,     6,    42,    16,    10,  1539,   528,     7, 27949,  4484,
           743,    77,   878, 12980,  5588,   420,  1533, 29829,   131,  3680,
             6,     5,  3783,  3471,   

# Section 4- Fine-tuning - Part 1

In [15]:
"""""
After the input is set, the pretrained model from HuggingFace is prepared in the following steps in part 1:
- model loaded for further fine-tuning with optional gpu acceleration,
- a new classification head is set for a new downstream task to classify 4 classes,
- and lastly pretrained model summary is printed.
"""""
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification

device = 'cuda' if torch.cuda.is_available() else 'cpu'


model = AutoModelForSequenceClassification.from_pretrained(
    "fakespot-ai/roberta-base-ai-text-detection-v1",
    num_labels=4,
    ignore_mismatched_sizes=True)

# Prints model summary
from torchinfo import summary

summary(model, input_size=(1, 512), dtypes=[torch.long])

config.json:   0%|          | 0.00/848 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at fakespot-ai/roberta-base-ai-text-detection-v1 and are newly initialized because the shapes did not match:
- classifier.out_proj.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.out_proj.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([4, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Layer (type:depth-idx)                                            Output Shape              Param #
RobertaForSequenceClassification                                  [1, 4]                    --
├─RobertaModel: 1-1                                               [1, 512, 768]             --
│    └─RobertaEmbeddings: 2-1                                     [1, 512, 768]             --
│    │    └─Embedding: 3-1                                        [1, 512, 768]             38,603,520
│    │    └─Embedding: 3-2                                        [1, 512, 768]             768
│    │    └─Embedding: 3-3                                        [1, 512, 768]             394,752
│    │    └─LayerNorm: 3-4                                        [1, 512, 768]             1,536
│    │    └─Dropout: 3-5                                          [1, 512, 768]             --
│    └─RobertaEncoder: 2-2                                        [1, 512, 768]             --
│    │    └─ModuleList: 3-6 

In [16]:
"""""
Change the flag to "True" only to proceed fine-tuning, otherwise in the next coming sectoin it loads the save model config from previous session.
"""""

IS_FINE_TUNING_ON = False  # or TRUE

In [17]:
# Fine-tuning - Part 2

# REFERENCE: https://huggingface.co/transformers/v3.2.0/custom_datasets.html

"""""
In part 2 the model is prepared for the actual fine-tuning process as follows:
- pretrained model is loaded into device (whether cpu or gpu if available),
- train_loader is initialized to stream and shuffle the dataset in mini-batches of 16,
- the AdamW optimizer is instantiated with a 5e-5 learning rate to update model parameters.

- the fine tuning process goes through 3 epochs where:
                        - total loss is zerod to..
                        -
                        -
                        -
"""""
if (IS_FINE_TUNING_ON):
    model.to(device)
    model.train()

    train_loader = DataLoader(X_train, batch_size=16, shuffle=True)

    optimiser = AdamW(model.parameters(), lr=5e-5)

    print("Fine tuning started...")

    for epoch in range(3):

        total_loss = 0.0
        for batch_idx, batch in enumerate(train_loader):
            optimiser.zero_grad()

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

            loss = outputs.loss

            loss.backward()
            optimiser.step()

            total_loss += loss.item()

            if batch_idx % 50 ==0:
                print(f"Epoch: {epoch+1} | batch: {batch_idx}/{len(train_loader)} | Running batch loss: {loss.item():.4f}")

    model.eval()
    print("Fine tuning is completed.")


In [18]:
# Saves the model weights and the tokenizer configurations locally
if (IS_FINE_TUNING_ON):

    from src.utils.version_number_generator import get_next_version_number

    current_version_number = get_next_version_number()  # gets current version

    version = f"version_{current_version_number}"       # sets version number

    # Saves model and tokeniser configurations
    model.save_pretrained(f"../model/{version}/")
    tokenizer.save_pretrained(f"../model/{version}/")
    print(f"A new model version is saved and located at the 'model/{version}/' folder.")

# Otherwise it fetches previous model configurations from Google Drive
else:
    print("### Downloading model from Google Drive... ###")
    downloader.start_downloading_model()
    print(f"\n\nModel is fetched and located at the 'model/version_1/' folder.")


### Downloading model from Google Drive... ###


Downloading...
From (original): https://drive.google.com/uc?id=1eU0eQVm0--OCHz_OHsM80o7Mo0E85LCg
From (redirected): https://drive.google.com/uc?id=1eU0eQVm0--OCHz_OHsM80o7Mo0E85LCg&confirm=t&uuid=228320a6-fe85-4251-89fb-f1c72d288dc0
To: /content/roberta-ai-text-detector/model/version_1.zip
100%|██████████| 464M/464M [00:04<00:00, 104MB/s]




Model is fetched and located at the 'model/version_1/' folder.


# Section 5 - Model Evaluation

In [19]:

# Initialises the arguments for the Eval step

model = AutoModelForSequenceClassification.from_pretrained("../model/version_1/")  # Loads saved trained model

test_loader = DataLoader(X_test, batch_size=16, shuffle=False)                     # Initialises the DataLoader with test set

from src.test_evaluator import TestEvaluator                                       # Imports the TestEvaluator class



evaluator = TestEvaluator(test_loader, model, device)                              # Initialises the evaluator with arguments from above

evaluator.calculate_metrics()                                                      # Starts calculating metrics including: Accuracy, Precision, Recall, F1 score

Evaluating the X_test dataset..
 EVALUATION COMPLETE | OVERALL ACCURACY: 95.83%
                              precision    recall  f1-score   support

Pure Human Written (Class 0)     0.9821    0.9125    0.9460       240
   Pure AI Written (Class 1)     0.9115    0.9875    0.9480       240
       AI Polished (Class 2)     1.0000    0.9375    0.9677       240
         Humanised (Class 3)     0.9484    0.9958    0.9715       240

                    accuracy                         0.9583       960
                   macro avg     0.9605    0.9583    0.9583       960
                weighted avg     0.9605    0.9583    0.9583       960



In [20]:
# Section 6 - Single Inference with Confidence Thresholding

# REFERENCE: https://docs.pytorch.org/docs/2.13/generated/torch.where.html

# Custom text for inference
raw_text_input = """ I really don't feel going to the shop tomorrow. The fact that it is Monday and there is a lot to do, it is just overwhelming."""

cleaned_text=text_preprocessor.clean_text(raw_text_input)

input_encodings = tokenizer([cleaned_text], truncation=True, padding="max_length", max_length=512, return_tensors="pt")

model.eval()

inputs = {
    'input_ids': input_encodings['input_ids'].to(device),
    'attention_mask': input_encodings['attention_mask'].to(device)
}

with torch.no_grad():
    outputs = model(**inputs)


# Pulls the highest probability class
probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

max_probs, argmax_classes = torch.max(probs, dim=1)

# If probability is lower or equal to 0.65, it keeps the class, otherwise returns class 4 (Unsure).
final_predictions = torch.where(max_probs >= 0.65, argmax_classes, 4)


# 7. Maps final probabilities back to classes
class_labels_mapping = {
    0: "Pure Human Written",
    1: "Pure AI Written",
    2: "Human Text Rewritten by AI (Polished)",
    3: "AI Text Rewritten by Human (Humanised)",
    4: "Unsure (Below 65% Confidence Boundary Threshold)"
}

predicted_id = final_predictions.item()
confidence_percentage = max_probs.item()

print("CUSTOM THRESHOLD SINGLE INFERENCE ANALYSIS REPORT\n")

print(f"Evaluated Model Confidence : {confidence_percentage:.2%}")
print(f"Final Categorized Prediction: Class {predicted_id} : {class_labels_mapping[predicted_id]}")


CUSTOM THRESHOLD SINGLE INFERENCE ANALYSIS REPORT

Evaluated Model Confidence : 99.77%
Final Categorized Prediction: Class 0 : Pure Human Written


In [ ]:
# Section 7 - Visualisations